In [1]:
import os
import glob
import chess.pgn
import pandas as pd
from tqdm import tqdm
import pyarrow

class ChessDataParser:
    def __init__(self, pgn_folder):
        self.pgn_folder = pgn_folder
        self.pgn_data = []
    
    def convert_result(self, result_str):
        if result_str == "1-0":
            return 1
        elif result_str == "0-1":
            return 0
        elif result_str == "1/2-1/2":
            return 0.5
        return None
    
    def load_pgn_data(self, max_games=None):
        print("♟️ Loading PGN files from folder...")
        pgn_files = glob.glob(os.path.join(self.pgn_folder, "*.pgn"))
        total_loaded = 0
        
        for file_path in pgn_files:
            print(f"   Parsing: {os.path.basename(file_path)}")
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as pgn:
                game_counter = 0
                while True:
                    game = chess.pgn.read_game(pgn)
                    if game is None:
                        break
                    
                    result = self.convert_result(game.headers.get("Result", "*"))
                    if result is None:
                        continue
                    
                    try:
                        white_elo = int(game.headers.get("WhiteElo", 0))
                        black_elo = int(game.headers.get("BlackElo", 0))
                    except ValueError:
                        continue
                    
                    game_length = len(list(game.mainline_moves()))
                    eco = game.headers.get("ECO", "UNKNOWN")
                    opening = game.headers.get("Opening", "UNKNOWN")
                    
                    self.pgn_data.append({
                        'white_elo': white_elo,
                        'black_elo': black_elo,
                        'result': result,
                        'game_length': game_length,
                        'eco': eco,
                        'opening': opening,
                        'source': os.path.basename(file_path)
                    })
                    
                    game_counter += 1
                    total_loaded += 1
                    
                    if max_games is not None and total_loaded >= max_games:
                        break
                        
            if max_games is not None and total_loaded >= max_games:
                break
                
        print(f"Loaded {total_loaded} games from {len(pgn_files)} PGN files")
    
    def get_dataframe(self):
        return pd.DataFrame(self.pgn_data)

def save_features_to_disk(dataframe, output_path, method="feather"):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    if method == "feather":
        dataframe.reset_index(drop=True).to_feather(output_path)
        print(f"Features saved to: {output_path} (format: feather)")
    elif method == "pkl":
        dataframe.to_pickle(output_path)
        print(f"Features saved to: {output_path} (format: pickle)")
    else:
        raise ValueError("Supported formats: 'feather' or 'pkl'")

In [2]:
# Parse PGN files and save data
pgn_folder = "C:/Users/chall/Documents/chess_game_pred/data/professional"
output_file = "C:/Users/chall/Documents/chess_game_pred/data/parsed/chess_games.feather"

parser = ChessDataParser(pgn_folder)
parser.load_pgn_data()  # Add max_games=50000 if testing
df = parser.get_dataframe()
save_features_to_disk(df, output_file, method="feather")

# Display basic info about the dataset
print(f"\nDataset shape: {df.shape}")
print(f"Result distribution:\n{df['result'].value_counts()}")

♟️ Loading PGN files from folder...
   Parsing: Aachen1868.pgn
   Parsing: Altona1869.pgn
   Parsing: Altona1872.pgn
   Parsing: Amsterdam1889.pgn
   Parsing: Amsterdam1920.pgn
   Parsing: Amsterdam1936.pgn
   Parsing: Amsterdam1976.pgn
   Parsing: Amsterdam1977.pgn
   Parsing: Amsterdam1978.pgn
   Parsing: Amsterdam1979.pgn
   Parsing: Amsterdam1980.pgn
   Parsing: Amsterdam1981.pgn
   Parsing: Amsterdam1985.pgn
   Parsing: Amsterdam1987.pgn
   Parsing: Amsterdam1988.pgn
   Parsing: Amsterdam1991.pgn
   Parsing: Amsterdam1993.pgn
   Parsing: Amsterdam1994.pgn
   Parsing: Amsterdam1995.pgn
   Parsing: Amsterdam1996.pgn
   Parsing: Astana2001.pgn
   Parsing: AVRO1938.pgn
   Parsing: Bad1977.pgn
   Parsing: BadElster1937.pgn
   Parsing: BadElster1938.pgn
   Parsing: BadElster1939.pgn
   Parsing: Baden1870.pgn
   Parsing: Baden1925.pgn
   Parsing: Baden1980.pgn
   Parsing: Baden2015.pgn
   Parsing: Baden2018.pgn
   Parsing: Baden2019.pgn
   Parsing: BadHarzburg1938.pgn
   Parsing: BadHarz

illegal san: 'Bf4' in 8/8/R1p5/1p1k4/1Pr5/1K6/8/8 w - - 18 115 while parsing <Game at 0x210875bf2f0 ('Caruana, Fabiano' vs. 'Carlsen, Magnus', '2018.11.09' at 'London ENG')>
illegal san: 'd4' in 8/8/R1p5/1p1k4/1Pr5/1K6/8/8 w - - 18 115 while parsing <Game at 0x210875bf2f0 ('Caruana, Fabiano' vs. 'Carlsen, Magnus', '2018.11.09' at 'London ENG', 1 errors)>


   Parsing: WorldChamp2021.pgn
   Parsing: WorldChamp2023.pgn
   Parsing: WorldChamp2024.pgn
   Parsing: WorldCup2005.pgn
   Parsing: WorldCup2007.pgn
   Parsing: WorldCup2009.pgn
   Parsing: WorldCup2011.pgn
   Parsing: WorldCup2013.pgn
   Parsing: WorldCup2015.pgn
   Parsing: WorldCup2023.pgn
   Parsing: Yerevan1965.pgn
   Parsing: Zagreb1965.pgn
   Parsing: Zagreb2019.pgn
   Parsing: Zandvoort1936.pgn
   Parsing: Zug2013.pgn
   Parsing: Zurich1934.pgn
   Parsing: Zurich2014.pgn
   Parsing: Zurich2015.pgn
Loaded 41741 games from 973 PGN files
Features saved to: C:/Users/chall/Documents/chess_game_pred/data/parsed/chess_games.feather (format: feather)

Dataset shape: (41741, 7)
Result distribution:
result
0.5    22172
1.0    12073
0.0     7496
Name: count, dtype: int64


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, classification_report, confusion_matrix, f1_score
import xgboost as xgb
import joblib

In [4]:
# =====================
# CONFIG
# =====================
DATA_PATH   = r"C:\Users\chall\Documents\chess_game_pred\data\parsed\chess_games.feather"
MODEL_PATH  = "models/chess_model.joblib"
PRED_CSV    = "models/validation_preds.csv"

WHITE_ELO   = "white_elo"
BLACK_ELO   = "black_elo"
RESULT_COL  = "result"
OPENING_COLS= ["opening_eco", "opening_name"]

CLASS_ORDER = ["white_win", "draw", "black_win"]

In [5]:
# =====================
# Load & preprocess data
# =====================
def load_data(path):
    df = pd.read_feather(path)

    # Map results to standard labels
    def to_result_label(x):
        if pd.isna(x): return np.nan
        if isinstance(x, (int,float)):
            if x==1.0: return "white_win"
            if x==0.0: return "black_win"
            if x==0.5: return "draw"
        key = str(x).strip().lower().replace("½","1/2")
        if key in ["1-0","w","white","white win"]: return "white_win"
        if key in ["0-1","b","black","black win"]: return "black_win"
        if key in ["1/2-1/2","draw","d","1/2"]: return "draw"
        return np.nan

    df["__result__"] = df[RESULT_COL].apply(to_result_label)
    df = df.dropna(subset=["__result__"])

    # Elo features safely
    df = df.assign(
        white_elo = df["white_elo"].replace(0, np.nan),
        black_elo = df["black_elo"].replace(0, np.nan)
    )
    df.dropna(subset=["white_elo","black_elo"], inplace=True)

    df = df.assign(
        elo_diff = df["white_elo"] - df["black_elo"],
        elo_diff_sq = (df["white_elo"] - df["black_elo"])**2,
        elo_ratio = (df["white_elo"] / df["black_elo"]).replace([np.inf,-np.inf], np.nan)
    )
    df.dropna(subset=["elo_ratio"], inplace=True)
    df["elo_ratio"] = df["elo_ratio"].clip(0.5, 2.0)

    return df.reset_index(drop=True)

In [ ]:
# =====================
# Leakage-safe target encoding for openings
# =====================
def target_encode_openings(df, cat_cols, y_col="__result__", n_splits=5):
    df_encoded = df.copy()
    for col in cat_cols:
        if col not in df.columns: continue
        for cls in CLASS_ORDER:
            df_encoded[f"{col}_prob_{cls}"] = 0.0

        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        for train_idx, val_idx in skf.split(df, df[y_col]):
            train, val = df.iloc[train_idx], df.iloc[val_idx]
            mapping = train.groupby(col)[y_col].value_counts(normalize=True).unstack().reindex(columns=CLASS_ORDER).fillna(0)
            for cls in CLASS_ORDER:
                df_encoded.loc[val_idx, f"{col}_prob_{cls}"] = val[col].map(mapping[cls]).fillna(mapping[cls].mean())
    return df_encoded

In [7]:
# =====================
# Prepare features
# =====================
def prepare_features(df):
    cat_cols = [c for c in OPENING_COLS if c in df.columns]
    df_enc = target_encode_openings(df, cat_cols)

    numeric_cols = [WHITE_ELO, BLACK_ELO, "elo_diff", "elo_diff_sq", "elo_ratio"]
    X_num = df_enc[numeric_cols].values
    te_cols = [c for c in df_enc.columns if any(c.startswith(col+"_prob_") for col in cat_cols)]
    X_te = df_enc[te_cols].values if te_cols else np.empty((len(df_enc),0))

    X = np.hstack([X_num, X_te])
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, df_enc

In [8]:
# =====================
# Train binary XGBoost
# =====================
def train_binary_xgb(X_train, y_train, X_valid, y_valid):
    model = xgb.XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        use_label_encoder=False,
        tree_method="hist",
        max_depth=6,
        n_estimators=300,
        learning_rate=0.1,
        random_state=42
    )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_valid)[:,1]
    y_pred = (y_proba >= 0.5).astype(int)
    return model, y_pred, y_proba

In [9]:
# =====================
# MAIN TRAINING PIPELINE
# =====================
def main():
    df = load_data(DATA_PATH)
    print(f"Loaded {len(df)} games from {DATA_PATH}")

    # Split train/validation
    df_train, df_valid = train_test_split(df, test_size=0.2, random_state=42, stratify=df["__result__"])

    # Prepare features
    X_train, df_train_enc = prepare_features(df_train)
    X_valid, df_valid_enc = prepare_features(df_valid)

    # -------------------
    # Step 1: Draw vs Non-Draw
    # -------------------
    print("\n=== STEP 1: Draw vs Non-Draw ===")
    y_train_step1 = (df_train_enc["__result__"]=="draw").astype(int)
    y_valid_step1 = (df_valid_enc["__result__"]=="draw").astype(int)

    model_step1, pred_step1, proba_step1 = train_binary_xgb(X_train, y_train_step1, X_valid, y_valid_step1)
    
    print(f"Step 1 Accuracy: {accuracy_score(y_valid_step1, pred_step1):.4f}")
    print(f"Step 1 F1: {f1_score(y_valid_step1, pred_step1):.4f}")

    # -------------------
    # Step 2: White vs Black (non-draw)
    # -------------------
    print("\n=== STEP 2: White vs Black (non-draw only) ===")
    non_draw_train_mask = df_train_enc["__result__"]!="draw"
    non_draw_valid_mask = df_valid_enc["__result__"]!="draw"

    X_train_step2 = X_train[non_draw_train_mask]
    y_train_step2 = (df_train_enc.loc[non_draw_train_mask,"__result__"]=="white_win").astype(int)

    X_valid_step2 = X_valid[non_draw_valid_mask]
    y_valid_step2 = (df_valid_enc.loc[non_draw_valid_mask,"__result__"]=="white_win").astype(int)

    # Check class distribution
    print("Step 2 training labels distribution:", np.bincount(y_train_step2))
    print("Step 2 validation labels distribution:", np.bincount(y_valid_step2))

    if len(np.unique(y_train_step2)) < 2:
        print("⚠️ Warning: Only one class present in Step 2 training. Skipping Step 2.")
        model_step2, pred_step2, proba_step2 = None, np.zeros_like(y_valid_step2), np.zeros_like(y_valid_step2,dtype=float)
    else:
        model_step2, pred_step2, proba_step2 = train_binary_xgb(X_train_step2, y_train_step2, X_valid_step2, y_valid_step2)
        print(f"Step 2 Accuracy: {accuracy_score(y_valid_step2, pred_step2):.4f}")
        print(f"Step 2 F1: {f1_score(y_valid_step2, pred_step2):.4f}")

    # -------------------
    # Combine predictions
    # -------------------
    print("\n=== COMBINING PREDICTIONS ===")
    final_pred = np.zeros(len(df_valid_enc), dtype=int)
    final_proba = np.zeros((len(df_valid_enc),3))

    # Draws
    draw_mask = pred_step1==1
    final_pred[draw_mask] = 1
    final_proba[draw_mask] = [0.0, 1.0, 0.0]

    # Non-draws
    non_draw_indices = np.flatnonzero(non_draw_valid_mask)
    if model_step2 is not None:
        final_pred[non_draw_indices] = np.where(pred_step2==1, 0, 2)
        final_proba[non_draw_indices] = np.column_stack([
            np.where(pred_step2==1,1.0,0.0),
            np.zeros(len(pred_step2)),
            np.where(pred_step2==0,1.0,0.0)
        ])
    else:
        # fallback: predict all as white
        final_pred[non_draw_indices] = 0
        final_proba[non_draw_indices] = [1.0,0.0,0.0]

    # -------------------
    # Fix any all-zero rows in proba
    # -------------------
    row_sums = final_proba.sum(axis=1)
    bad_rows = np.where(row_sums == 0)[0]
    if len(bad_rows) > 0:
        print(f"Warning: {len(bad_rows)} rows had zero probabilities. Fixing...")
        final_proba[bad_rows, 1] = 1.0   # mark as draw
        final_pred[bad_rows] = 1

    # -------------------
    # Evaluation
    # -------------------
    print("\n=== FINAL RESULTS ===")
    y_true = df_valid_enc["__result__"].map({"white_win":0,"draw":1,"black_win":2}).values

    acc = accuracy_score(y_true, final_pred)
    loss = log_loss(y_true, final_proba)
    macro_f1 = f1_score(y_true, final_pred, average="macro")
    
    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation Log Loss: {loss:.4f}")
    print(f"Validation Macro-F1: {macro_f1:.4f}\n")
    
    print("Classification Report:")
    print(classification_report(y_true, final_pred, target_names=CLASS_ORDER))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, final_pred))

    # -------------------
    # Save models + predictions
    # -------------------
    os.makedirs("models", exist_ok=True)
    joblib.dump({"step1": model_step1, "step2": model_step2}, MODEL_PATH)
    print(f"\nModels saved to {MODEL_PATH}")

    valid_out = pd.DataFrame({
        "y_true": y_true,
        "y_pred": final_pred,
        "proba_white_win": final_proba[:,0],
        "proba_draw": final_proba[:,1],
        "proba_black_win": final_proba[:,2]
    })
    valid_out.to_csv(PRED_CSV, index=False)
    print(f"Validation predictions saved to {PRED_CSV}")
    
    return model_step1, model_step2, valid_out

# Run the main training pipeline
model_step1, model_step2, predictions_df = main()

Loaded 41437 games from C:\Users\chall\Documents\chess_game_pred\data\parsed\chess_games.feather

=== STEP 1: Draw vs Non-Draw ===


c:\Users\chall\Documents\chess_game_pred\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:07:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Step 1 Accuracy: 0.5705
Step 1 F1: 0.6423

=== STEP 2: White vs Black (non-draw only) ===
Step 2 training labels distribution: [5949 9595]
Step 2 validation labels distribution: [1487 2399]


c:\Users\chall\Documents\chess_game_pred\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [21:07:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Step 2 Accuracy: 0.6873
Step 2 F1: 0.7683

=== COMBINING PREDICTIONS ===

=== FINAL RESULTS ===
Validation Accuracy: 0.8534
Validation Log Loss: 5.2839
Validation Macro-F1: 0.7626

Classification Report:
              precision    recall  f1-score   support

   white_win       0.71      0.84      0.77      2399
        draw       1.00      1.00      1.00      4402
   black_win       0.63      0.44      0.52      1487

    accuracy                           0.85      8288
   macro avg       0.78      0.76      0.76      8288
weighted avg       0.85      0.85      0.85      8288


Confusion Matrix:
[[2014    0  385]
 [   0 4402    0]
 [ 830    0  657]]

Models saved to models/chess_model.joblib
Validation predictions saved to models/validation_preds.csv


In [10]:
# Display prediction results
print("First 10 predictions:")
print(predictions_df.head(10))

print(f"\nPrediction distribution:")
print(predictions_df['y_pred'].value_counts())

print(f"\nTrue label distribution:")
print(predictions_df['y_true'].value_counts())

First 10 predictions:
   y_true  y_pred  proba_white_win  proba_draw  proba_black_win
0       0       0              1.0         0.0              0.0
1       1       1              0.0         1.0              0.0
2       0       0              1.0         0.0              0.0
3       1       1              0.0         1.0              0.0
4       2       0              1.0         0.0              0.0
5       1       1              0.0         1.0              0.0
6       0       0              1.0         0.0              0.0
7       1       1              0.0         1.0              0.0
8       1       1              0.0         1.0              0.0
9       1       1              0.0         1.0              0.0

Prediction distribution:
y_pred
1    4402
0    2844
2    1042
Name: count, dtype: int64

True label distribution:
y_true
1    4402
0    2399
2    1487
Name: count, dtype: int64
